# sharklane -- Saleh Bay-style workflow with a vectorized habitat + the global shipping lane

This notebook runs the mitigation pipeline using:
- a **core habitat polygon loaded from `/content/high_value_pixels.geojson`**
  (e.g. output of raster threshold vectorization -- "high value" whale
  shark density pixels turned into polygons)
- the **bundled global shipping lane dataset** (Major/Middle/Minor) instead
  of a custom-drawn or AIS-derived lane

Upload `high_value_pixels.geojson` to Colab's `/content/` first (drag it
into the Files pane on the left, or `from google.colab import files;
files.upload()`).


In [ ]:
# Install sharklane. Pick ONE of the following depending on where the
# package currently lives (uncomment the one you need):

# Option A -- published to PyPI:
# !pip install -q sharklane

# Option B -- installing straight from a GitHub repo (replace with your URL):
# !pip install -q git+https://github.com/YOUR_USERNAME/sharklane.git

# Option C -- you uploaded the wheel file to /content/ yourself:
# !pip install -q /content/sharklane-0.2.0-py3-none-any.whl

!pip install -q geopandas shapely rasterio scikit-image matplotlib pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sharklane import Simulator

HABITAT_PATH = "/content/high_value_pixels.geojson"

## 1. Inspect the input file and pick a working CRS

The habitat file could be in any CRS (commonly WGS84 / EPSG:4326 if it
came from a lon/lat raster). We need a *projected* CRS in metres for all
the distance/area math downstream -- this cell auto-picks the correct UTM
zone from the data's own centroid, so you don't have to hardcode one.

In [ ]:
raw = gpd.read_file(HABITAT_PATH)
print(f"{len(raw)} feature(s) loaded, CRS: {raw.crs}")

if raw.crs is None:
    # no CRS metadata in the file -- assume WGS84 (lon/lat), the most common
    # case for raster-derived vector output. Adjust this if you know otherwise.
    raw = raw.set_crs("EPSG:4326")
    print("No CRS found in file -- assumed EPSG:4326 (lon/lat).")

raw_wgs84 = raw.to_crs("EPSG:4326")
centroid = raw_wgs84.union_all().centroid
lon, lat = centroid.x, centroid.y

utm_zone = int((lon + 180) / 6) + 1
hemisphere_code = 32600 if lat >= 0 else 32700  # WGS84 UTM north/south EPSG base
working_crs = f"EPSG:{hemisphere_code + utm_zone}"

print(f"Centroid: {lon:.4f}, {lat:.4f}  ->  auto-picked working CRS: {working_crs}")

## 2. Handle multiple disjoint polygons ("high value pixels" -> possibly many patches)

If the file has several separate high-value patches, merging them directly
gives a `MultiPolygon`, which the reroute/lane-shift geometry functions
can't use directly (they need a single `Polygon` with a defined boundary).

Three reasonable options, pick the one that fits your case:
- **`"convex_hull"`** (default here): wraps everything in the smallest
  convex polygon that contains all patches. Simple, always works, but may
  overstate the risk area if patches are spread out.
- **`"largest"`**: just use the single biggest patch, ignore the rest.
  Simple, but silently drops area.
- **`"buffer_merge"`**: buffer each patch outward by some distance and
  merge overlapping buffers into one shape, then remove the buffer. Keeps
  the patches' actual shape better than convex_hull, but needs a distance
  parameter tuned to your data's gap sizes.

In [ ]:
MERGE_STRATEGY = "convex_hull"  # "convex_hull" | "largest" | "buffer_merge"
BUFFER_MERGE_DISTANCE_M = 2000    # only used if MERGE_STRATEGY == "buffer_merge"

habitat_proj = raw.to_crs(working_crs)
merged = habitat_proj.geometry.union_all()

if merged.geom_type == "MultiPolygon":
    print(f"Input has {len(merged.geoms)} disjoint polygons -- "
          f"applying merge strategy: {MERGE_STRATEGY}")
    if MERGE_STRATEGY == "convex_hull":
        habitat_polygon = merged.convex_hull
    elif MERGE_STRATEGY == "largest":
        habitat_polygon = max(merged.geoms, key=lambda g: g.area)
    elif MERGE_STRATEGY == "buffer_merge":
        habitat_polygon = merged.buffer(BUFFER_MERGE_DISTANCE_M).buffer(-BUFFER_MERGE_DISTANCE_M)
        if habitat_polygon.geom_type == "MultiPolygon":
            print("Still disjoint after buffer_merge -- try a larger BUFFER_MERGE_DISTANCE_M, "
                  "or fall back to convex_hull.")
            habitat_polygon = habitat_polygon.convex_hull
    else:
        raise ValueError("MERGE_STRATEGY must be 'convex_hull', 'largest', or 'buffer_merge'")
else:
    habitat_polygon = merged
    print("Input is already a single connected polygon -- no merging needed.")

print(f"Final habitat polygon area: {habitat_polygon.area / 1e6:.2f} km^2")

## 3. Load the habitat into the Simulator

In [ ]:
sim = Simulator(working_crs=working_crs)

# write the resolved single-polygon habitat back out and load it through
# the normal API, so sim.polygon / sim.polygon_gdf are set up consistently
resolved_gdf = gpd.GeoDataFrame(geometry=[habitat_polygon], crs=working_crs)
resolved_gdf.to_file("/content/_resolved_habitat.geojson", driver="GeoJSON")
sim.load_core_habitat("/content/_resolved_habitat.geojson", source_crs=working_crs)

print("Habitat loaded:", sim.polygon.area / 1e6, "km^2")

## 4. Use the GLOBAL shipping lane (bundled dataset) instead of a custom one

`load_world_shipping_lane()` pulls from the bundled Major/Middle/Minor
dataset, clips to a padded box around your habitat, and picks the segment
closest to the habitat centroid. Try `lane_type="Major"` or `"Minor"` too --
if none is found nearby, widen `pad_deg`.

Remember: this is a coarse, hand-digitized global dataset (a handful of
features worldwide), good for orientation and as a fallback -- not a
substitute for real local AIS once you have it.

In [ ]:
line = sim.load_world_shipping_lane(lane_type="Middle", pad_deg=2.0)
print(f"Loaded global lane, length: {line.length / 1000:.1f} km")

## 5. Load AIS traffic

The mitigation simulations (speed reduction, rerouting, lane shift) all
run against actual vessel tracks -- you need this step, they don't work on
habitat/lane geometry alone.

**If you have real AIS**, load it here instead:
```python
sim.load_ais("/content/your_ais.csv", min_points=20, stationary_frac_limit=0.25)
```
(CSV needs `vessel_id, timestamp, lon, lat` columns.)

**No real AIS yet?** The cell below builds SYNTHETIC vessels that follow
the global lane you just loaded, purely so the rest of this notebook has
something to run against. Clearly a placeholder -- swap it for real data
before drawing any actual conclusions about your site.

In [ ]:
USE_SYNTHETIC_AIS = True  # set False once you have real AIS to load instead

if USE_SYNTHETIC_AIS:
    rng = np.random.default_rng(0)
    x0, y0 = sim.corridor_line.coords[0]
    x1, y1 = sim.corridor_line.coords[-1]

    rows = []
    t0 = pd.Timestamp("2024-08-01")
    n_vessels = 25
    for v in range(n_vessels):
        frac_offset = rng.uniform(-0.15, 0.15)
        speed_knots = rng.uniform(10, 20)
        speed_mps = speed_knots * 0.514
        n_pts = rng.integers(30, 60)
        t_start = t0 + pd.Timedelta(hours=int(rng.integers(0, 24 * 60)))

        xs = np.linspace(x0, x1, n_pts) + rng.normal(0, 300, n_pts)
        ys = (np.linspace(y0, y1, n_pts) + frac_offset * (y1 - y0)
              + rng.normal(0, 300, n_pts))
        seg_dist = np.hypot(np.diff(xs), np.diff(ys))
        seg_hours = seg_dist / speed_mps / 3600
        cum_hours = np.concatenate([[0], np.cumsum(seg_hours)])

        for i in range(n_pts):
            rows.append({"vessel_id": f"V{v:03d}",
                         "timestamp": t_start + pd.Timedelta(hours=cum_hours[i]),
                         "x": xs[i], "y": ys[i]})

    ais_df = pd.DataFrame(rows).sort_values(["vessel_id", "timestamp"]).reset_index(drop=True)
    ais_gdf = gpd.GeoDataFrame(ais_df, geometry=gpd.points_from_xy(ais_df["x"], ais_df["y"]),
                                crs=working_crs)

    from sharklane.ais import clean_tracks, to_tracks
    cleaned = clean_tracks(ais_gdf, crs_metric=working_crs, max_speed_mps=30,
                            min_points=10, stationary_frac_limit=0.9)
    sim.tracks = to_tracks(cleaned)
    print(f"Synthetic vessels loaded: {len(sim.tracks)}")
else:
    sim.load_ais("/content/your_ais.csv", min_points=20, stationary_frac_limit=0.25)
    print(f"Real AIS loaded: {len(sim.tracks)} vessels")

## 6. (Optional but recommended) Water/land mask for rerouting + lane shift

Speed reduction doesn't need this, but rerouting (least-cost pathing) and
lane-shift feasibility do, since they need to know what's actually
navigable water. Colab has full internet access, so we can fetch a real
coastline (Natural Earth 10m) on the fly and clip it to the area.

In [ ]:
# fetch Natural Earth 10m coastline (via a GitHub mirror) and clip to a
# padded box around the habitat + lane
!wget -q -O /content/_ne_coastline.shp "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/10m_physical/ne_10m_coastline.shp"
!wget -q -O /content/_ne_coastline.shx "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/10m_physical/ne_10m_coastline.shx"
!wget -q -O /content/_ne_coastline.dbf "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/10m_physical/ne_10m_coastline.dbf"
!wget -q -O /content/_ne_coastline.prj "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/10m_physical/ne_10m_coastline.prj"

coastline = gpd.read_file("/content/_ne_coastline.shp")

habitat_wgs84_bounds = gpd.GeoSeries([sim.polygon], crs=working_crs).to_crs("EPSG:4326").total_bounds
lane_wgs84_bounds = gpd.GeoSeries([sim.corridor_line], crs=working_crs).to_crs("EPSG:4326").total_bounds
pad = 2.0
minx = min(habitat_wgs84_bounds[0], lane_wgs84_bounds[0]) - pad
miny = min(habitat_wgs84_bounds[1], lane_wgs84_bounds[1]) - pad
maxx = max(habitat_wgs84_bounds[2], lane_wgs84_bounds[2]) + pad
maxy = max(habitat_wgs84_bounds[3], lane_wgs84_bounds[3]) + pad

from shapely.geometry import box as shapely_box, Polygon
clipped = coastline[coastline.intersects(shapely_box(minx, miny, maxx, maxy))]

# coastline features are closed rings -- turn them into land polygons
land_polys = [Polygon(geom.coords) for geom in clipped.geometry if geom.coords[0] == geom.coords[-1]]
if not land_polys:
    print("No closed coastline rings found in this area -- if this region has "
          "no nearby islands/mainland, you can skip the mask and just run "
          "speed reduction below (rerouting/lane-shift need this step).")
else:
    land_gdf = gpd.GeoDataFrame(geometry=land_polys, crs="EPSG:4326")
    land_gdf.to_file("/content/_land.geojson", driver="GeoJSON")

    minx_p, miny_p, maxx_p, maxy_p = sim.polygon.bounds
    lminx, lminy, lmaxx, lmaxy = sim.corridor_line.bounds
    bounds_proj = (min(minx_p, lminx) - 15000, min(miny_p, lminy) - 15000,
                   max(maxx_p, lmaxx) + 15000, max(maxy_p, lmaxy) + 15000)

    sim.build_mask("/content/_land.geojson", bounds=bounds_proj, resolution=200,
                    source_crs="EPSG:4326")
    print(f"Mask built: {sim.water_mask.shape}, water fraction: {sim.water_mask.mask.mean():.2f}")

## 7. Run the three mitigation simulations

In [ ]:
# omit reductions= to use the library default: 10-75% at 1% increments
# (matching Womersley et al. 2024's method) -- gives a smooth curve.
speed_summary = sim.simulate_speed_reduction()
print(speed_summary)

In [ ]:
if sim.water_mask is not None:
    try:
        reroute_time = sim.simulate_redirection(method="least_cost")
        print(sim.results["redirection"]["labels"])
        print(reroute_time)
    except ValueError as e:
        print(f"Redirection unavailable: {e}\n"
              "Likely cause: no vessel tracks actually enter/exit the "
              "habitat polygon in this dataset -- check vessel classification "
              "or widen the habitat/lane setup.")
else:
    print("Skipped -- no water mask built (see step 6).")

In [ ]:
if sim.water_mask is not None:
    # NOTE: this can legitimately fail with "does not cross the polygon
    # boundary twice" -- the global lane is just the NEAREST segment to
    # the habitat centroid, not guaranteed to actually pass through it.
    # That's real information (this lane isn't a relevant transit route
    # through your core habitat), not a bug -- handle it explicitly rather
    # than letting it crash the notebook.
    try:
        opts = sim.list_reroute_options(side="west")
        print(opts)
    except ValueError as e:
        print(f"Reroute options unavailable: {e}\n"
              "This usually means the auto-picked global lane doesn't "
              "actually cross your habitat polygon -- try a different "
              "lane_type in step 4, or use a custom/AIS-derived lane "
              "that you know passes through the area.")
else:
    print("Skipped -- no water mask built (see step 6).")

In [ ]:
if sim.water_mask is not None:
    lane_result = sim.simulate_lane_shift(speed_threshold_mps=5.0, direction=(0, 1),
                                           offsets_m=list(range(0, 20000, 1000)))
    print("Min feasible offset (m):", lane_result.get("min_feasible_offset_m"))
else:
    print("Skipped -- no water mask built (see step 6).")

## 8. Static graphics

In [ ]:
ax = sim.plot_site_map(target_polygon=sim.polygon, title="Habitat + global shipping lane")
plt.show()

In [ ]:
ax2 = sim.plot_speed_reduction_curve()
plt.show()

In [ ]:
if sim.water_mask is not None:
    ax3 = sim.plot_reroute_paths(target_polygon=sim.polygon)
    plt.show()

out_path = "/content/transit_comparison.gif"

# NOTE: if the global lane doesn't actually cross the habitat polygon (see
# the reroute-options cell above), the 'reroute' scenario can't be built --
# fall back to just baseline + speed_reduction in that case rather than
# crashing the whole animation.
scenarios_to_animate = ["baseline", "speed_reduction", "reroute"]
try:
    sim.animate_transit_comparison(
        side="west", reduction=0.6, out_path=out_path,
        n_frames=100, fps=15, show_time_chart=True,
        scenarios=scenarios_to_animate,
    )
except ValueError as e:
    print(f"Full 3-way comparison unavailable ({e}); "
          "falling back to baseline + speed_reduction only.")
    scenarios_to_animate = ["baseline", "speed_reduction"]
    sim.animate_transit_comparison(
        side="west", reduction=0.6, out_path=out_path,
        n_frames=100, fps=15, show_time_chart=True,
        scenarios=scenarios_to_animate,
    )

display(Image(filename=out_path))

In [ ]:
out_path = "/content/transit_comparison.gif"

# NOTE: if the global lane doesn't actually cross the habitat polygon (see
# the reroute-options cell above), the 'reroute' scenario can't be built --
# fall back to just baseline + speed_reduction in that case rather than
# crashing the whole animation.
scenarios_to_animate = ["baseline", "speed_reduction", "reroute"]
try:
    sim.animate_transit_comparison(
        side="west", reduction=0.6, out_path=out_path,
        n_frames=100, fps=15, show_time_chart=True,
        scenarios=scenarios_to_animate,
    )
except ValueError as e:
    print(f"Full 3-way comparison unavailable ({e}); "
          "falling back to baseline + speed_reduction only.")
    scenarios_to_animate = ["baseline", "speed_reduction"]
    sim.animate_transit_comparison(
        side="west", reduction=0.6, out_path=out_path,
        n_frames=100, fps=15, show_time_chart=True,
        scenarios=scenarios_to_animate,
    )

display(Image(filename=out_path))

## Notes

- If your habitat area has no nearby coastline (fully open ocean), step
  6's mask will legitimately come back empty -- that's fine, it just means
  rerouting/lane-shift aren't meaningfully different from open water in
  this area; speed reduction still applies regardless.
- Re-run step 4 with a different `lane_type` or larger `pad_deg` if no
  global lane is found nearby.
- Set `USE_SYNTHETIC_AIS = False` in step 5 once you have real AIS --
  everything downstream is identical either way.
